# Базовый baseline (логистическая регрессия на HOG)

In [1]:
import numpy as np
import cv2
from skimage.feature import hog
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import os
from pathlib import Path
from tqdm import tqdm

## Пути

In [2]:
TRAIN_DIR = Path("../data/processed/train")
VAL_DIR = Path("../data/processed/val")
TEST_DIR = Path("../data/raw/seg_test/seg_test")
classes = sorted([d.name for d in TRAIN_DIR.iterdir() if d.is_dir()])

In [3]:
def extract_hog_features(img_path, size=(64,64)):
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, size)
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    features = hog(gray, orientations=9, pixels_per_cell=(8,8),
                   cells_per_block=(2,2), visualize=False)
    return features

## Загрузка признаков для train и val (может занять время)

In [4]:
X_train, y_train = [], []
for i, cls in enumerate(classes):
    cls_dir = TRAIN_DIR / cls
    for img_path in tqdm(list(cls_dir.glob("*.jpg")), desc=cls):
        feats = extract_hog_features(img_path)
        X_train.append(feats)
        y_train.append(i)

X_val, y_val = [], []
for i, cls in enumerate(classes):
    cls_dir = VAL_DIR / cls
    for img_path in tqdm(list(cls_dir.glob("*.jpg")), desc=cls):
        feats = extract_hog_features(img_path)
        X_val.append(feats)
        y_val.append(i)

buildings:   0%|          | 0/1752 [00:00<?, ?it/s]

street: 100%|██████████| 477/477 [00:07<00:00, 62.33it/s]


## Обучение логистической регрессии

In [5]:
clf = LogisticRegression(max_iter=1000, random_state=42)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_val)
print("Validation accuracy:", accuracy_score(y_val, y_pred))
print(classification_report(y_val, y_pred, target_names=classes))

Validation accuracy: 0.6551601423487544
              precision    recall  f1-score   support

   buildings       0.68      0.69      0.69       439
      forest       0.83      0.87      0.85       455
     glacier       0.54      0.55      0.55       481
    mountain       0.52      0.53      0.53       503
         sea       0.59      0.57      0.58       455
      street       0.77      0.74      0.76       477

    accuracy                           0.66      2810
   macro avg       0.66      0.66      0.66      2810
weighted avg       0.65      0.66      0.65      2810

